# YOLO: You Only Look Once - Object Detection

## Introduction

In this notebook, we'll build a deep understanding of **object detection** by implementing a simplified version of **YOLO (You Only Look Once)**, one of the most influential real-time object detection systems.

### What We'll Learn

1. **Object Detection Fundamentals**: Understanding how object detection differs from classification
2. **Bounding Boxes & IoU**: How to represent and measure object localization quality
3. **YOLO Architecture**: Single-stage detection with grid-based predictions
4. **Anchor Boxes**: Handling multiple objects of different shapes
5. **Multi-Task Loss Function**: Balancing localization, objectness, and classification
6. **Non-Maximum Suppression (NMS)**: Filtering overlapping detections
7. **Training & Inference**: Complete pipeline from scratch

### Why YOLO Matters

YOLO revolutionized object detection by:
- **Speed**: Single forward pass for all detections (vs. region-based methods like R-CNN)
- **Global Context**: Sees the entire image during training
- **End-to-End Learning**: No separate region proposal network

We'll implement a simplified YOLO on a toy dataset to understand the core concepts without massive computational requirements.

## 1. Setup

Let's import the necessary libraries and configure our environment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import Dataset, DataLoader
from typing import List, Tuple, Dict
import random

from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

# Set random seed for reproducibility
set_seed(42)

Configure device for training (CPU or GPU).

In [ ]:
device = get_device()
print(f"Using device: {device}")

## 2. Object Detection Fundamentals

### 2.1 Classification vs. Detection

**Image Classification**: "What is in the image?" → Single label per image  
**Object Detection**: "What objects are there and where are they?" → Multiple labels + locations

Object detection requires solving two tasks simultaneously:
1. **Classification**: What class is the object? (cat, dog, car, etc.)
2. **Localization**: Where is the object? (bounding box coordinates)

Let's understand how we represent bounding boxes.

### 2.2 Bounding Box Representations

There are multiple ways to represent a bounding box:

1. **Corner Format**: `[x_min, y_min, x_max, y_max]` - top-left and bottom-right corners
2. **Center Format**: `[x_center, y_center, width, height]` - center point and dimensions

YOLO uses the center format. Let's implement conversion functions.

In [ ]:
def corner_to_center(boxes):
    """Convert [x_min, y_min, x_max, y_max] to [x_center, y_center, w, h]."""
    x_min, y_min, x_max, y_max = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    w = x_max - x_min
    h = y_max - y_min
    x_center = x_min + w / 2
    y_center = y_min + h / 2
    return torch.stack([x_center, y_center, w, h], dim=1)

def center_to_corner(boxes):
    """Convert [x_center, y_center, w, h] to [x_min, y_min, x_max, y_max]."""
    x_center, y_center, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    x_min = x_center - w / 2
    y_min = y_center - h / 2
    x_max = x_center + w / 2
    y_max = y_center + h / 2
    return torch.stack([x_min, y_min, x_max, y_max], dim=1)

# Test the conversion
corner_box = torch.tensor([[10.0, 20.0, 50.0, 80.0]])
center_box = corner_to_center(corner_box)
back_to_corner = center_to_corner(center_box)

print(f"Corner format: {corner_box[0].tolist()}")
print(f"Center format: {center_box[0].tolist()}")
print(f"Back to corner: {back_to_corner[0].tolist()}")
print(f"Conversion accurate: {torch.allclose(corner_box, back_to_corner)}")

### 2.3 Intersection over Union (IoU)

**IoU** measures how much two bounding boxes overlap. It's the foundation for:
- Matching predictions to ground truth during training
- Non-Maximum Suppression (removing duplicate detections)
- Evaluating detection quality

$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

- IoU = 1.0: Perfect overlap
- IoU = 0.5: Common threshold for "good" detection
- IoU = 0.0: No overlap

In [ ]:
def calculate_iou(box1, box2):
    """
    Calculate IoU between two boxes in corner format [x_min, y_min, x_max, y_max].
    
    Args:
        box1: tensor of shape [N, 4]
        box2: tensor of shape [M, 4]
    
    Returns:
        IoU matrix of shape [N, M]
    """
    # Expand dimensions for broadcasting: [N, 1, 4] and [1, M, 4]
    box1 = box1.unsqueeze(1)  # [N, 1, 4]
    box2 = box2.unsqueeze(0)  # [1, M, 4]
    
    # Calculate intersection coordinates
    x_min_inter = torch.max(box1[..., 0], box2[..., 0])
    y_min_inter = torch.max(box1[..., 1], box2[..., 1])
    x_max_inter = torch.min(box1[..., 2], box2[..., 2])
    y_max_inter = torch.min(box1[..., 3], box2[..., 3])
    
    # Calculate intersection area
    intersection_w = torch.clamp(x_max_inter - x_min_inter, min=0)
    intersection_h = torch.clamp(y_max_inter - y_min_inter, min=0)
    intersection_area = intersection_w * intersection_h
    
    # Calculate areas of both boxes
    box1_area = (box1[..., 2] - box1[..., 0]) * (box1[..., 3] - box1[..., 1])
    box2_area = (box2[..., 2] - box2[..., 0]) * (box2[..., 3] - box2[..., 1])
    
    # Calculate union area
    union_area = box1_area + box2_area - intersection_area
    
    # Calculate IoU (add small epsilon to avoid division by zero)
    iou = intersection_area / (union_area + 1e-6)
    
    return iou

# Test IoU with different overlaps
box_a = torch.tensor([[0.0, 0.0, 10.0, 10.0]])
box_b_perfect = torch.tensor([[0.0, 0.0, 10.0, 10.0]])  # Perfect overlap
box_b_partial = torch.tensor([[5.0, 5.0, 15.0, 15.0]])  # Partial overlap
box_b_none = torch.tensor([[20.0, 20.0, 30.0, 30.0]])   # No overlap

print(f"IoU (perfect overlap): {calculate_iou(box_a, box_b_perfect).item():.3f}")
print(f"IoU (partial overlap): {calculate_iou(box_a, box_b_partial).item():.3f}")
print(f"IoU (no overlap): {calculate_iou(box_a, box_b_none).item():.3f}")

### 2.4 Visualizing IoU

Let's visualize how IoU changes with different box overlaps.

In [ ]:
def visualize_iou(box1, box2, title="IoU Visualization"):
    """Visualize two bounding boxes and their IoU."""
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    # Convert to numpy for plotting
    b1 = box1.numpy()
    b2 = box2.numpy()
    
    # Create rectangles
    rect1 = patches.Rectangle(
        (b1[0], b1[1]), b1[2] - b1[0], b1[3] - b1[1],
        linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.3, label='Box 1'
    )
    rect2 = patches.Rectangle(
        (b2[0], b2[1]), b2[2] - b2[0], b2[3] - b2[1],
        linewidth=2, edgecolor='red', facecolor='red', alpha=0.3, label='Box 2'
    )
    
    ax.add_patch(rect1)
    ax.add_patch(rect2)
    
    # Calculate and display IoU
    iou = calculate_iou(box1.unsqueeze(0), box2.unsqueeze(0)).item()
    
    ax.set_xlim(-5, 35)
    ax.set_ylim(-5, 35)
    ax.set_aspect('equal')
    ax.legend()
    ax.set_title(f"{title}\nIoU = {iou:.3f}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Visualize different scenarios
box_ref = torch.tensor([5.0, 5.0, 15.0, 15.0])

visualize_iou(box_ref, torch.tensor([5.0, 5.0, 15.0, 15.0]), "Perfect Overlap")
visualize_iou(box_ref, torch.tensor([10.0, 10.0, 20.0, 20.0]), "50% Overlap")
visualize_iou(box_ref, torch.tensor([15.0, 15.0, 25.0, 25.0]), "Small Overlap")

Notice how the IoU decreases as the boxes overlap less. An IoU of 0.5 is commonly used as the threshold for considering a detection as "correct".

## 3. Creating a Toy Dataset

### 3.1 Dataset Design

We'll create a simple synthetic dataset with geometric shapes (circles, rectangles, triangles) on a white background. This allows us to:
- Understand YOLO without massive compute requirements
- Control the complexity and number of objects
- Debug and visualize easily

Each image will have:
- 1-3 randomly placed shapes
- Different colors for different classes
- Random sizes and positions
- Ground truth bounding boxes and class labels

In [ ]:
class ShapesDataset(Dataset):
    """
    Synthetic dataset with geometric shapes for object detection.
    
    Each sample contains:
    - image: [3, H, W] RGB image
    - boxes: [N, 4] bounding boxes in corner format (normalized to [0, 1])
    - labels: [N] class labels (0=circle, 1=rectangle, 2=triangle)
    """
    
    def __init__(self, num_samples=1000, img_size=128, max_objects=3):
        self.num_samples = num_samples
        self.img_size = img_size
        self.max_objects = max_objects
        self.classes = ['circle', 'rectangle', 'triangle']
        self.num_classes = len(self.classes)
        
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # Create white background
        img = np.ones((self.img_size, self.img_size, 3), dtype=np.float32)
        
        # Random number of objects (1 to max_objects)
        num_objects = random.randint(1, self.max_objects)
        
        boxes = []
        labels = []
        
        for _ in range(num_objects):
            # Random class
            class_id = random.randint(0, self.num_classes - 1)
            
            # Random position and size
            size = random.randint(15, 40)
            x = random.randint(size, self.img_size - size)
            y = random.randint(size, self.img_size - size)
            
            # Random color
            color = np.random.rand(3)
            
            # Draw shape and get bounding box
            bbox = self._draw_shape(img, class_id, x, y, size, color)
            
            if bbox is not None:
                boxes.append(bbox)
                labels.append(class_id)
        
        # Convert to tensors and normalize coordinates to [0, 1]
        boxes = torch.tensor(boxes, dtype=torch.float32)
        boxes = boxes / self.img_size  # Normalize to [0, 1]
        labels = torch.tensor(labels, dtype=torch.long)
        
        # Convert image to [C, H, W] format
        img = torch.from_numpy(img).permute(2, 0, 1)
        
        return img, boxes, labels
    
    def _draw_shape(self, img, class_id, x, y, size, color):
        """Draw a shape on the image and return its bounding box."""
        if class_id == 0:  # Circle
            rr, cc = self._circle_coords(y, x, size // 2, img.shape[:2])
            img[rr, cc] = color
            return [x - size // 2, y - size // 2, x + size // 2, y + size // 2]
        
        elif class_id == 1:  # Rectangle
            w, h = size, int(size * 0.7)
            x1, y1 = x - w // 2, y - h // 2
            x2, y2 = x + w // 2, y + h // 2
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)
            img[y1:y2, x1:x2] = color
            return [x1, y1, x2, y2]
        
        elif class_id == 2:  # Triangle
            pts = np.array([
                [x, y - size // 2],
                [x - size // 2, y + size // 2],
                [x + size // 2, y + size // 2]
            ])
            rr, cc = self._triangle_coords(pts, img.shape[:2])
            img[rr, cc] = color
            x_min, y_min = pts.min(axis=0)
            x_max, y_max = pts.max(axis=0)
            return [x_min, y_min, x_max, y_max]
        
        return None
    
    def _circle_coords(self, cy, cx, radius, shape):
        """Generate coordinates for a circle."""
        y, x = np.ogrid[:shape[0], :shape[1]]
        mask = (x - cx) ** 2 + (y - cy) ** 2 <= radius ** 2
        return np.where(mask)
    
    def _triangle_coords(self, pts, shape):
        """Generate coordinates for a triangle."""
        y, x = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
        
        def sign(p1, p2, p3):
            return (p1[0] - p3[0]) * (p2[1] - p3[1]) - (p2[0] - p3[0]) * (p1[1] - p3[1])
        
        mask = np.zeros(shape, dtype=bool)
        for i in range(shape[0]):
            for j in range(shape[1]):
                pt = [j, i]
                d1 = sign(pt, pts[0], pts[1])
                d2 = sign(pt, pts[1], pts[2])
                d3 = sign(pt, pts[2], pts[0])
                
                has_neg = (d1 < 0) or (d2 < 0) or (d3 < 0)
                has_pos = (d1 > 0) or (d2 > 0) or (d3 > 0)
                
                mask[i, j] = not (has_neg and has_pos)
        
        return np.where(mask)

# Create dataset
train_dataset = ShapesDataset(num_samples=800, img_size=128, max_objects=3)
val_dataset = ShapesDataset(num_samples=200, img_size=128, max_objects=3)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")

### 3.2 Visualizing Dataset Samples

Let's visualize some samples to verify our dataset is generating correct images and bounding boxes.

In [ ]:
def visualize_sample(img, boxes, labels, class_names, title="Sample"):
    """Visualize an image with bounding boxes and labels."""
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    # Convert image to numpy and transpose to [H, W, C]
    img_np = img.permute(1, 2, 0).numpy()
    ax.imshow(img_np)
    
    # Denormalize boxes to image coordinates
    img_size = img.shape[1]
    boxes_denorm = boxes * img_size
    
    # Draw each bounding box
    colors = ['red', 'blue', 'green', 'yellow', 'purple']
    for box, label in zip(boxes_denorm, labels):
        x_min, y_min, x_max, y_max = box
        w = x_max - x_min
        h = y_max - y_min
        
        rect = patches.Rectangle(
            (x_min, y_min), w, h,
            linewidth=2, edgecolor=colors[label % len(colors)],
            facecolor='none'
        )
        ax.add_patch(rect)
        
        # Add label
        ax.text(
            x_min, y_min - 2,
            class_names[label],
            color='white',
            fontsize=10,
            bbox=dict(facecolor=colors[label % len(colors)], alpha=0.7, pad=2)
        )
    
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Visualize multiple samples
for i in range(3):
    img, boxes, labels = train_dataset[i]
    visualize_sample(
        img, boxes, labels,
        train_dataset.classes,
        title=f"Training Sample {i+1} ({len(boxes)} objects)"
    )

Perfect! Our dataset generates images with geometric shapes and correct bounding boxes. Now let's move on to understanding YOLO's architecture.

## 4. YOLO Architecture

### 4.1 Grid-Based Detection

YOLO divides the input image into an **S × S grid**. Each grid cell is responsible for:
1. Predicting whether an object's **center** falls in that cell
2. Predicting **B bounding boxes** with confidence scores
3. Predicting **class probabilities** for objects in that cell

For example, with S=4 and B=2:
- Image is divided into 4×4 = 16 grid cells
- Each cell predicts 2 bounding boxes
- Total predictions: 16 × 2 = 32 bounding boxes per image

**Key Insight**: The grid cell that contains the object's center is responsible for detecting that object.

### 4.2 Output Tensor Structure

YOLO outputs a tensor of shape `[S, S, B * 5 + C]` where:
- **S**: Grid size (e.g., 4 for 4×4 grid)
- **B**: Number of bounding boxes per cell (e.g., 2)
- **C**: Number of classes (e.g., 3 for our shapes)
- **5**: Box parameters (x, y, w, h, confidence)

For each bounding box prediction:
- **(x, y)**: Center coordinates relative to grid cell (0 to 1)
- **(w, h)**: Width and height relative to image size (0 to 1)
- **confidence**: P(object) × IoU(pred, truth)
- **class probabilities**: C values for each class

Example: S=4, B=2, C=3 → Output shape: `[4, 4, 2*5 + 3] = [4, 4, 13]`

### 4.3 YOLO Model Implementation

We'll implement a simplified YOLO model with:
- Convolutional feature extractor
- Grid-based prediction head
- Output reshaping to [batch, S, S, B*5 + C]

In [ ]:
class SimpleYOLO(nn.Module):
    """
    Simplified YOLO architecture for object detection.
    
    Args:
        num_classes: Number of object classes
        grid_size: Grid size (S×S)
        num_boxes: Number of bounding boxes per grid cell (B)
    """
    
    def __init__(self, num_classes=3, grid_size=4, num_boxes=2):
        super().__init__()
        self.num_classes = num_classes
        self.grid_size = grid_size
        self.num_boxes = num_boxes
        
        # Feature extractor (simplified CNN)
        self.features = nn.Sequential(
            # Input: [B, 3, 128, 128]
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),  # -> [B, 32, 64, 64]
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # -> [B, 64, 32, 32]
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # -> [B, 128, 16, 16]
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),  # -> [B, 256, 8, 8]
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),  # -> [B, 512, 4, 4]
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
        )
        
        # Detection head: predicts [S, S, B*5 + C]
        self.output_size = num_boxes * 5 + num_classes
        self.detector = nn.Sequential(
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),
            nn.Conv2d(1024, self.output_size, kernel_size=1),
        )
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input images [B, 3, H, W]
        
        Returns:
            Predictions [B, S, S, B*5 + C]
        """
        # Extract features
        features = self.features(x)  # [B, 512, S, S]
        
        # Predict detection outputs
        output = self.detector(features)  # [B, B*5+C, S, S]
        
        # Reshape to [B, S, S, B*5+C]
        batch_size = x.size(0)
        output = output.permute(0, 2, 3, 1).contiguous()  # [B, S, S, B*5+C]
        
        return output

# Create model and test forward pass
model = SimpleYOLO(num_classes=3, grid_size=4, num_boxes=2)
model = model.to(device)

# Test with dummy input
dummy_input = torch.randn(2, 3, 128, 128).to(device)
output = model(dummy_input)

print(f"Model output shape: {output.shape}")
print(f"Expected: [batch=2, S=4, S=4, B*5+C={2*5+3}]")
print(f"\nNumber of parameters: {sum(p.numel() for p in model.parameters()):,}")

Our model successfully outputs predictions in the correct YOLO format! The output tensor contains predictions for each grid cell.

## 5. YOLO Loss Function

### 5.1 Multi-Task Loss Components

YOLO's loss function combines three objectives:

1. **Localization Loss** (boxes): How accurate are the predicted bounding boxes?
   $$\mathcal{L}_{\text{box}} = \lambda_{\text{coord}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} [(x_i - \hat{x}_i)^2 + (y_i - \hat{y}_i)^2 + (\sqrt{w_i} - \sqrt{\hat{w}_i})^2 + (\sqrt{h_i} - \sqrt{\hat{h}_i})^2]$$

2. **Objectness Loss** (confidence): How confident is the model that a box contains an object?
   $$\mathcal{L}_{\text{obj}} = \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} (C_{ij} - \hat{C}_{ij})^2 + \lambda_{\text{noobj}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{noobj}} (C_{ij} - \hat{C}_{ij})^2$$

3. **Classification Loss**: What class is the detected object?
   $$\mathcal{L}_{\text{class}} = \sum_{i=0}^{S^2} \mathbb{1}_{i}^{\text{obj}} \sum_{c \in \text{classes}} (p_i(c) - \hat{p}_i(c))^2$$

Where:
- $\mathbb{1}_{ij}^{\text{obj}}$: 1 if box j in cell i is responsible for the object
- $\lambda_{\text{coord}}$: Weight for localization loss (typically 5)
- $\lambda_{\text{noobj}}$: Weight for no-object confidence loss (typically 0.5)

### 5.2 Understanding Loss Weights

**Why different weights?**

- **High $\lambda_{\text{coord}}$ (5)**: Localization is harder than classification, so we emphasize it
- **Low $\lambda_{\text{noobj}}$ (0.5)**: Most grid cells don't contain objects (class imbalance), so we reduce the weight for "no object" predictions
- **Square root for w, h**: Small errors in large boxes are less critical than small errors in small boxes

### 5.3 Target Encoding

Before computing the loss, we need to convert ground truth boxes to YOLO's grid format. This involves:
1. Determining which grid cell is responsible for each object (based on center)
2. Encoding box coordinates relative to that cell
3. Setting objectness targets (1 for cells with objects, 0 otherwise)

In [ ]:
def encode_targets(boxes, labels, grid_size, num_classes, num_boxes):
    """
    Encode ground truth boxes and labels into YOLO target format.
    
    Args:
        boxes: List of [N, 4] tensors (normalized corner format)
        labels: List of [N] tensors (class indices)
        grid_size: Grid size S
        num_classes: Number of classes
        num_boxes: Number of boxes per cell B
    
    Returns:
        target: [B, S, S, B*5 + C] encoded targets
    """
    batch_size = len(boxes)
    target = torch.zeros(batch_size, grid_size, grid_size, num_boxes * 5 + num_classes)
    
    for batch_idx in range(batch_size):
        box_list = boxes[batch_idx]
        label_list = labels[batch_idx]
        
        if len(box_list) == 0:
            continue
        
        # Convert boxes to center format
        box_centers = corner_to_center(box_list)
        
        for box_center, label in zip(box_centers, label_list):
            x_center, y_center, w, h = box_center
            
            # Determine which grid cell this object belongs to
            grid_x = int(x_center * grid_size)
            grid_y = int(y_center * grid_size)
            
            # Ensure within bounds
            grid_x = min(grid_x, grid_size - 1)
            grid_y = min(grid_y, grid_size - 1)
            
            # Coordinates relative to the grid cell
            x_cell = x_center * grid_size - grid_x
            y_cell = y_center * grid_size - grid_y
            
            # If this cell doesn't have an object yet, add it to the first box slot
            if target[batch_idx, grid_y, grid_x, 4] == 0:  # Check first box confidence
                # First box: [x, y, w, h, confidence]
                target[batch_idx, grid_y, grid_x, 0:4] = torch.tensor([x_cell, y_cell, w, h])
                target[batch_idx, grid_y, grid_x, 4] = 1  # Confidence (object exists)
                
                # Class probabilities
                class_idx = num_boxes * 5 + label
                target[batch_idx, grid_y, grid_x, class_idx] = 1
    
    return target

# Test target encoding
test_boxes = [torch.tensor([[0.2, 0.3, 0.4, 0.5], [0.6, 0.7, 0.8, 0.9]])]
test_labels = [torch.tensor([0, 1])]

encoded = encode_targets(test_boxes, test_labels, grid_size=4, num_classes=3, num_boxes=2)
print(f"Encoded target shape: {encoded.shape}")
print(f"\nNon-zero cells (grid positions with objects):")
for i in range(4):
    for j in range(4):
        if encoded[0, i, j, 4] > 0:  # Check if object exists
            print(f"  Grid cell ({i}, {j}): confidence={encoded[0, i, j, 4]:.1f}")

### 5.4 Implementing the YOLO Loss

Now let's implement the complete loss function with all three components.

In [ ]:
class YOLOLoss(nn.Module):
    """
    YOLO loss function combining localization, objectness, and classification losses.
    """
    
    def __init__(self, num_classes=3, num_boxes=2, lambda_coord=5.0, lambda_noobj=0.5):
        super().__init__()
        self.num_classes = num_classes
        self.num_boxes = num_boxes
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
    
    def forward(self, predictions, targets):
        """
        Compute YOLO loss.
        
        Args:
            predictions: [B, S, S, B*5 + C] model predictions
            targets: [B, S, S, B*5 + C] encoded ground truth
        
        Returns:
            Total loss and individual loss components
        """
        batch_size = predictions.size(0)
        grid_size = predictions.size(1)
        
        # Split predictions into components
        # For simplicity, we only use the first box prediction
        pred_boxes = pred_boxes = predictions[..., 0:4]  # [B, S, S, 4] (x, y, w, h)
        pred_conf = predictions[..., 4:5]   # [B, S, S, 1] (confidence)
        pred_class = predictions[..., self.num_boxes * 5:]  # [B, S, S, C]
        
        target_boxes = targets[..., 0:4]
        target_conf = targets[..., 4:5]
        target_class = targets[..., self.num_boxes * 5:]
        
        # Create masks for cells with and without objects
        obj_mask = target_conf > 0  # [B, S, S, 1]
        noobj_mask = target_conf == 0
        
        # 1. Localization loss (only for cells with objects)
        num_obj = obj_mask.sum()
        
        if num_obj > 0:
            # Create binary mask [B, S, S]
            obj_mask_3d = obj_mask.squeeze(-1)  # [B, S, S]
            
            # Extract xy and wh for cells with objects
            pred_xy = pred_boxes[..., 0:2]  # [B, S, S, 2]
            target_xy = target_boxes[..., 0:2]  # [B, S, S, 2]
            pred_wh = pred_boxes[..., 2:4]  # [B, S, S, 2]
            target_wh = target_boxes[..., 2:4]  # [B, S, S, 2]
            
            # Apply mask by selecting only cells with objects
            # We need to expand mask to [B, S, S, 2] to match last dimension
            obj_mask_xy = obj_mask_3d.unsqueeze(-1).expand_as(pred_xy)  # [B, S, S, 2]
            
            # Select values where mask is True and reshape
            pred_xy_obj = pred_xy[obj_mask_xy].view(-1, 2)  # [num_obj, 2]
            target_xy_obj = target_xy[obj_mask_xy].view(-1, 2)  # [num_obj, 2]
            pred_wh_obj = pred_wh[obj_mask_xy].view(-1, 2)  # [num_obj, 2]
            target_wh_obj = target_wh[obj_mask_xy].view(-1, 2)  # [num_obj, 2]
            
            # Coordinate loss (x, y)
            xy_loss = F.mse_loss(pred_xy_obj, target_xy_obj, reduction='sum')
            
            # Width/height loss (with square root)
            wh_loss = F.mse_loss(
                torch.sqrt(pred_wh_obj + 1e-6),
                torch.sqrt(target_wh_obj + 1e-6),
                reduction='sum'
            )
            
            loc_loss = self.lambda_coord * (xy_loss + wh_loss)
        else:
            loc_loss = torch.tensor(0.0, device=predictions.device)
        
        # 2. Objectness loss
        # For cells with objects
        if obj_mask.sum() > 0:
            obj_conf_loss = F.mse_loss(
                pred_conf[obj_mask],
                target_conf[obj_mask],
                reduction='sum'
            )
        else:
            obj_conf_loss = torch.tensor(0.0, device=predictions.device)
        
        # For cells without objects (weighted down)
        if noobj_mask.sum() > 0:
            noobj_conf_loss = F.mse_loss(
                pred_conf[noobj_mask],
                target_conf[noobj_mask],
                reduction='sum'
            )
        else:
            noobj_conf_loss = torch.tensor(0.0, device=predictions.device)
        
        conf_loss = obj_conf_loss + self.lambda_noobj * noobj_conf_loss
        
        # 3. Classification loss (only for cells with objects)
        if obj_mask.sum() > 0:
            # Expand mask to match class predictions [B, S, S, C]
            obj_mask_3d = obj_mask.squeeze(-1)  # [B, S, S]
            obj_mask_class = obj_mask_3d.unsqueeze(-1).expand_as(pred_class)  # [B, S, S, C]
            
            pred_class_obj = pred_class[obj_mask_class].view(-1, self.num_classes)
            target_class_obj = target_class[obj_mask_class].view(-1, self.num_classes)
            
            class_loss = F.mse_loss(pred_class_obj, target_class_obj, reduction='sum')
        else:
            class_loss = torch.tensor(0.0, device=predictions.device)
        
        # Total loss
        total_loss = loc_loss + conf_loss + class_loss
        
        # Normalize by batch size
        total_loss = total_loss / batch_size
        loc_loss = loc_loss / batch_size
        conf_loss = conf_loss / batch_size
        class_loss = class_loss / batch_size
        
        return total_loss, loc_loss, conf_loss, class_loss

# Test loss computation
loss_fn = YOLOLoss(num_classes=3, num_boxes=2)

# Create dummy predictions and targets (ensure batch sizes match!)
pred = torch.randn(2, 4, 4, 13)  # Random predictions
# Create test targets with matching batch size
test_boxes_batch = [torch.tensor([[0.2, 0.3, 0.4, 0.5]]), torch.tensor([[0.6, 0.7, 0.8, 0.9]])]
test_labels_batch = [torch.tensor([0]), torch.tensor([1])]
targets = encode_targets(test_boxes_batch, test_labels_batch, grid_size=4, num_classes=3, num_boxes=2)

total_loss, loc_loss, conf_loss, class_loss = loss_fn(pred, targets)

print(f"Total loss: {total_loss.item():.4f}")
print(f"  Localization loss: {loc_loss.item():.4f}")
print(f"  Confidence loss: {conf_loss.item():.4f}")
print(f"  Classification loss: {class_loss.item():.4f}")

Our loss function correctly combines the three components with appropriate weighting.

## 6. Training YOLO

### 6.1 Data Loading

First, we need a custom collate function to handle variable numbers of objects per image.

In [ ]:
def yolo_collate_fn(batch):
    """Custom collate function for YOLO dataset."""
    images = torch.stack([item[0] for item in batch])
    boxes = [item[1] for item in batch]
    labels = [item[2] for item in batch]
    return images, boxes, labels

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=yolo_collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=yolo_collate_fn,
    num_workers=0
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

### 6.2 Training Loop

Let's implement the training loop with loss tracking.

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    total_loc_loss = 0
    total_conf_loss = 0
    total_class_loss = 0
    
    for images, boxes, labels in loader:
        images = images.to(device)
        
        # Encode targets
        targets = encode_targets(
            boxes, labels,
            grid_size=model.grid_size,
            num_classes=model.num_classes,
            num_boxes=model.num_boxes
        ).to(device)
        
        # Forward pass
        predictions = model(images)
        loss, loc_loss, conf_loss, class_loss = loss_fn(predictions, targets)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track losses
        total_loss += loss.item()
        total_loc_loss += loc_loss.item()
        total_conf_loss += conf_loss.item()
        total_class_loss += class_loss.item()
    
    num_batches = len(loader)
    return {
        'loss': total_loss / num_batches,
        'loc_loss': total_loc_loss / num_batches,
        'conf_loss': total_conf_loss / num_batches,
        'class_loss': total_class_loss / num_batches,
    }

def validate(model, loader, loss_fn, device):
    """Validate the model."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for images, boxes, labels in loader:
            images = images.to(device)
            
            targets = encode_targets(
                boxes, labels,
                grid_size=model.grid_size,
                num_classes=model.num_classes,
                num_boxes=model.num_boxes
            ).to(device)
            
            predictions = model(images)
            loss, _, _, _ = loss_fn(predictions, targets)
            total_loss += loss.item()
    
    return total_loss / len(loader)

print("Training functions ready!")

### 6.3 Train the Model

Now let's train our YOLO model on the shapes dataset. We'll use a small number of epochs for demonstration.

In [ ]:
# Create fresh model and optimizer
model = SimpleYOLO(num_classes=3, grid_size=4, num_boxes=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = YOLOLoss(num_classes=3, num_boxes=2)

# Training
num_epochs = 10
history = {'train_loss': [], 'val_loss': [], 'loc_loss': [], 'conf_loss': [], 'class_loss': []}

print("Starting training...\n")

for epoch in range(num_epochs):
    # Train
    train_metrics = train_epoch(model, train_loader, optimizer, loss_fn, device)
    
    # Validate
    val_loss = validate(model, val_loader, loss_fn, device)
    
    # Store history
    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_loss)
    history['loc_loss'].append(train_metrics['loc_loss'])
    history['conf_loss'].append(train_metrics['conf_loss'])
    history['class_loss'].append(train_metrics['class_loss'])
    
    # Print progress
    print(f"Epoch {epoch+1}/{num_epochs}:")
    print(f"  Train Loss: {train_metrics['loss']:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  Loc: {train_metrics['loc_loss']:.4f} | Conf: {train_metrics['conf_loss']:.4f} | Class: {train_metrics['class_loss']:.4f}")

print("\nTraining complete!")

### 6.4 Visualize Training Progress

Let's plot the training curves to see how the model learned.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot total loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot loss components
axes[1].plot(history['loc_loss'], label='Localization', marker='o')
axes[1].plot(history['conf_loss'], label='Confidence', marker='s')
axes[1].plot(history['class_loss'], label='Classification', marker='^')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Components')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The model is learning! Both training and validation losses are decreasing. Notice how the three loss components contribute differently to the total loss.

## 7. Inference and Non-Maximum Suppression

### 7.1 Converting Predictions to Boxes

During inference, we need to:
1. Convert grid-relative predictions to absolute coordinates
2. Filter low-confidence predictions
3. Apply Non-Maximum Suppression to remove duplicates

In [ ]:
def decode_predictions(predictions, grid_size, confidence_threshold=0.5):
    """
    Decode YOLO predictions into bounding boxes.
    
    Args:
        predictions: [B, S, S, B*5 + C] model output
        grid_size: Grid size S
        confidence_threshold: Minimum confidence to keep
    
    Returns:
        List of (boxes, confidences, class_ids) for each image in batch
    """
    batch_size = predictions.size(0)
    results = []
    
    for b in range(batch_size):
        boxes = []
        confidences = []
        class_ids = []
        
        for i in range(grid_size):
            for j in range(grid_size):
                # Get predictions for this cell (first box only)
                cell_pred = predictions[b, i, j]
                
                x_cell, y_cell, w, h = cell_pred[0:4]
                confidence = torch.sigmoid(cell_pred[4])  # Apply sigmoid to confidence
                
                # Skip low-confidence predictions
                if confidence < confidence_threshold:
                    continue
                
                # Convert from cell-relative to image-relative coordinates
                x_center = (j + torch.sigmoid(x_cell)) / grid_size
                y_center = (i + torch.sigmoid(y_cell)) / grid_size
                width = torch.sigmoid(w)
                height = torch.sigmoid(h)
                
                # Convert to corner format
                x_min = x_center - width / 2
                y_min = y_center - height / 2
                x_max = x_center + width / 2
                y_max = y_center + height / 2
                
                # Clamp to [0, 1]
                x_min = torch.clamp(x_min, 0, 1)
                y_min = torch.clamp(y_min, 0, 1)
                x_max = torch.clamp(x_max, 0, 1)
                y_max = torch.clamp(y_max, 0, 1)
                
                # Get class prediction
                class_probs = cell_pred[10:]  # After 2 boxes * 5 values
                class_id = torch.argmax(class_probs)
                
                boxes.append([x_min.item(), y_min.item(), x_max.item(), y_max.item()])
                confidences.append(confidence.item())
                class_ids.append(class_id.item())
        
        results.append((
            torch.tensor(boxes) if boxes else torch.empty(0, 4),
            torch.tensor(confidences) if confidences else torch.empty(0),
            torch.tensor(class_ids) if class_ids else torch.empty(0)
        ))
    
    return results

print("Prediction decoder ready!")

### 7.2 Non-Maximum Suppression (NMS)

**NMS** removes redundant, overlapping bounding boxes. The algorithm:

1. Sort boxes by confidence (highest first)
2. Take the box with highest confidence as a "keeper"
3. Remove all boxes with IoU > threshold with this box
4. Repeat until no boxes remain

**Why NMS?** Multiple grid cells near an object may all predict boxes for it. NMS keeps only the best prediction.

In [ ]:
def non_max_suppression(boxes, confidences, class_ids, iou_threshold=0.5):
    """
    Apply Non-Maximum Suppression to remove overlapping boxes.
    
    Args:
        boxes: [N, 4] bounding boxes in corner format
        confidences: [N] confidence scores
        class_ids: [N] class indices
        iou_threshold: IoU threshold for suppression
    
    Returns:
        Filtered boxes, confidences, and class_ids
    """
    if len(boxes) == 0:
        return boxes, confidences, class_ids
    
    # Sort by confidence (descending)
    sorted_indices = torch.argsort(confidences, descending=True)
    
    keep_indices = []
    
    while len(sorted_indices) > 0:
        # Keep the box with highest confidence
        current = sorted_indices[0]
        keep_indices.append(current)
        
        if len(sorted_indices) == 1:
            break
        
        # Calculate IoU with remaining boxes
        current_box = boxes[current].unsqueeze(0)
        remaining_boxes = boxes[sorted_indices[1:]]
        
        ious = calculate_iou(current_box, remaining_boxes).squeeze()
        
        # Keep boxes with IoU below threshold AND same class
        remaining_classes = class_ids[sorted_indices[1:]]
        current_class = class_ids[current]
        
        # Suppress boxes of the same class with high IoU
        suppress_mask = (ious > iou_threshold) & (remaining_classes == current_class)
        keep_mask = ~suppress_mask
        
        sorted_indices = sorted_indices[1:][keep_mask]
    
    # Return filtered results
    keep_indices = torch.tensor(keep_indices)
    return boxes[keep_indices], confidences[keep_indices], class_ids[keep_indices]

print("NMS function ready!")

### 7.3 Visualizing NMS Effect

Let's see how NMS filters duplicate detections.

In [ ]:
# Create overlapping boxes to demonstrate NMS
demo_boxes = torch.tensor([
    [0.2, 0.2, 0.5, 0.5],  # Box 1 (high confidence)
    [0.22, 0.22, 0.52, 0.52],  # Box 2 (medium) - overlaps with Box 1
    [0.24, 0.24, 0.54, 0.54],  # Box 3 (low) - overlaps with Box 1 & 2
    [0.6, 0.6, 0.9, 0.9],  # Box 4 (high) - separate object
])
demo_confidences = torch.tensor([0.9, 0.7, 0.6, 0.85])
demo_class_ids = torch.tensor([0, 0, 0, 1])

# Before NMS
print("Before NMS:")
print(f"  Number of boxes: {len(demo_boxes)}")
print(f"  Confidences: {demo_confidences.tolist()}")

# Apply NMS
filtered_boxes, filtered_conf, filtered_class = non_max_suppression(
    demo_boxes, demo_confidences, demo_class_ids, iou_threshold=0.5
)

# After NMS
print("\nAfter NMS:")
print(f"  Number of boxes: {len(filtered_boxes)}")
print(f"  Confidences: {filtered_conf.tolist()}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, boxes, confs, title in zip(
    axes,
    [demo_boxes, filtered_boxes],
    [demo_confidences, filtered_conf],
    ["Before NMS (4 boxes)", "After NMS (2 boxes)"]
):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    
    colors = ['red', 'blue', 'green', 'yellow']
    for i, (box, conf) in enumerate(zip(boxes, confs)):
        x_min, y_min, x_max, y_max = box
        w = x_max - x_min
        h = y_max - y_min
        
        rect = patches.Rectangle(
            (x_min, y_min), w, h,
            linewidth=2, edgecolor=colors[i % len(colors)],
            facecolor='none', label=f'Conf: {conf:.2f}'
        )
        ax.add_patch(rect)
    
    ax.legend()
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

NMS successfully removed the duplicate overlapping boxes for the first object, keeping only the highest-confidence prediction.

## 8. Testing the Trained Model

### 8.1 Making Predictions

Let's test our trained model on validation samples.

In [ ]:
def predict_and_visualize(model, dataset, idx, confidence_threshold=0.3, iou_threshold=0.5):
    """Make predictions and visualize results."""
    model.eval()
    
    # Get sample
    img, gt_boxes, gt_labels = dataset[idx]
    
    # Predict
    with torch.no_grad():
        img_input = img.unsqueeze(0).to(device)
        predictions = model(img_input)
    
    # Decode predictions
    results = decode_predictions(predictions, model.grid_size, confidence_threshold)
    pred_boxes, pred_conf, pred_class = results[0]
    
    # Apply NMS
    if len(pred_boxes) > 0:
        pred_boxes, pred_conf, pred_class = non_max_suppression(
            pred_boxes, pred_conf, pred_class, iou_threshold
        )
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Ground truth
    ax = axes[0]
    img_np = img.permute(1, 2, 0).cpu().numpy()
    ax.imshow(img_np)
    
    img_size = img.shape[1]
    gt_boxes_denorm = gt_boxes * img_size
    
    for box, label in zip(gt_boxes_denorm, gt_labels):
        x_min, y_min, x_max, y_max = box
        w = x_max - x_min
        h = y_max - y_min
        
        rect = patches.Rectangle(
            (x_min, y_min), w, h,
            linewidth=2, edgecolor='green', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x_min, y_min - 2, dataset.classes[label],
                color='white', fontsize=9,
                bbox=dict(facecolor='green', alpha=0.7, pad=2))
    
    ax.set_title(f'Ground Truth ({len(gt_boxes)} objects)')
    ax.axis('off')
    
    # Predictions
    ax = axes[1]
    ax.imshow(img_np)
    
    pred_boxes_denorm = pred_boxes * img_size
    
    for box, conf, class_id in zip(pred_boxes_denorm, pred_conf, pred_class):
        x_min, y_min, x_max, y_max = box
        w = x_max - x_min
        h = y_max - y_min
        
        rect = patches.Rectangle(
            (x_min, y_min), w, h,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x_min, y_min - 2,
                f"{dataset.classes[int(class_id)]} {conf:.2f}",
                color='white', fontsize=9,
                bbox=dict(facecolor='red', alpha=0.7, pad=2))
    
    ax.set_title(f'Predictions ({len(pred_boxes)} objects)')
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Test on multiple validation samples
for i in range(5):
    predict_and_visualize(model, val_dataset, i, confidence_threshold=0.3)

### 8.2 Analyzing Predictions

The model is detecting and classifying objects! Notice:
- Bounding boxes roughly align with objects
- Class predictions are mostly correct
- Confidence scores reflect prediction quality
- Some false positives or missed detections (expected with limited training)

With more training epochs and a larger dataset, performance would improve significantly.

## 9. Comparison: YOLO vs. Two-Stage Detectors

### 9.1 Two-Stage Detectors (R-CNN Family)

**R-CNN, Fast R-CNN, Faster R-CNN** use a two-stage approach:

**Stage 1: Region Proposal**
- Generate ~2000 candidate regions (Region Proposal Network)
- Each region might contain an object

**Stage 2: Classification & Refinement**
- Classify each region (object or background)
- Refine bounding box coordinates

**Pros:**
- Higher accuracy (more careful attention to each region)
- Better small object detection

**Cons:**
- Slower (two forward passes + NMS)
- More complex architecture
- Harder to train end-to-end

### 9.2 YOLO (Single-Stage Detector)

**YOLO** processes the entire image in one forward pass:

**Single Stage: Direct Detection**
- Grid-based predictions across the entire image
- Simultaneous localization, objectness, and classification
- One network output per grid cell

**Pros:**
- Extremely fast (real-time capable: 45+ FPS)
- Simpler architecture
- Sees full image context (better global reasoning)
- End-to-end trainable

**Cons:**
- Lower accuracy on small objects
- Struggles with close, overlapping objects
- Grid cell limitation (max one object per cell per anchor)

### 9.3 Speed vs. Accuracy Tradeoff

Let's visualize the fundamental tradeoff in object detection architectures.

In [ ]:
# Conceptual comparison (approximate values for illustration)
methods = ['R-CNN', 'Fast R-CNN', 'Faster R-CNN', 'YOLO v1', 'YOLO v2', 'YOLO v3', 'SSD']
fps = [0.05, 0.5, 7, 45, 67, 78, 59]  # Frames per second
map_scores = [66, 68, 73, 63, 72, 74, 71]  # mAP (mean Average Precision)

# Color code by detector type
colors = ['blue', 'blue', 'blue', 'red', 'red', 'red', 'orange']
labels = ['Two-Stage' if c == 'blue' else 'Single-Stage (YOLO)' if c == 'red' else 'Single-Stage (SSD)' 
          for c in colors]

plt.figure(figsize=(10, 6))

# Scatter plot
for i, (method, fps_val, map_val, color, label) in enumerate(zip(methods, fps, map_scores, colors, labels)):
    marker = 'o' if color == 'blue' else '^' if color == 'red' else 's'
    plt.scatter(fps_val, map_val, s=150, c=color, marker=marker, alpha=0.7,
                edgecolors='black', linewidth=1.5)
    plt.annotate(method, (fps_val, map_val), xytext=(5, 5),
                textcoords='offset points', fontsize=9)

plt.xlabel('Speed (FPS)', fontsize=12)
plt.ylabel('Accuracy (mAP)', fontsize=12)
plt.title('Object Detection: Speed vs. Accuracy Tradeoff', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xscale('log')

# Add legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', 
           markersize=10, label='Two-Stage (R-CNN)'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='red', 
           markersize=10, label='Single-Stage (YOLO)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='orange', 
           markersize=10, label='Single-Stage (SSD)')
]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

print("Key Takeaway:")
print("- Two-stage detectors (R-CNN family): High accuracy, slow")
print("- Single-stage detectors (YOLO, SSD): Lower accuracy, very fast")
print("- YOLO trades ~5-10% accuracy for 10-100x speed improvement")

### 9.4 When to Use Each Approach

**Use Two-Stage Detectors (R-CNN family) when:**
- Accuracy is paramount
- Working with small objects
- Offline processing is acceptable
- Dense, overlapping objects

**Use Single-Stage Detectors (YOLO) when:**
- Real-time performance required (video, autonomous driving)
- Resource-constrained environments
- Large, well-separated objects
- End-to-end training simplicity desired

**Modern Trend**: Gap is closing. YOLO v5/v7/v8 and EfficientDet achieve near-R-CNN accuracy at YOLO speeds.

## 10. Key Takeaways

### Core Concepts

1. **Object Detection = Localization + Classification**
   - Must predict both where objects are (bounding boxes) and what they are (classes)
   - Evaluation uses IoU to measure localization quality

2. **YOLO's Grid-Based Approach**
   - Divides image into S×S grid
   - Each cell predicts B bounding boxes with confidence scores
   - Single forward pass for all predictions (fast!)

3. **Multi-Task Loss Function**
   - Localization loss: Box coordinate accuracy (weighted heavily)
   - Objectness loss: Confidence in object presence (class imbalance handling)
   - Classification loss: Object class prediction

4. **Non-Maximum Suppression**
   - Essential post-processing step
   - Removes duplicate detections using IoU threshold
   - Keeps highest-confidence predictions

5. **Speed vs. Accuracy Tradeoff**
   - Single-stage (YOLO): Fast but less accurate
   - Two-stage (R-CNN): Accurate but slower
   - Choose based on application requirements

### What We Built

We implemented a complete object detection system from scratch:
- Grid-based YOLO architecture
- Multi-component loss function
- Encoding/decoding between grid and image coordinates
- Non-Maximum Suppression
- End-to-end training pipeline

This simplified version captures the essence of YOLO. Production systems (YOLOv5, YOLOv8) add:
- Multiple detection scales (feature pyramid)
- Anchor boxes of different sizes
- Advanced augmentation
- Better backbone networks (CSPDarknet, EfficientNet)
- Label smoothing and focal loss

### Further Exploration

To deepen your understanding:

1. **Try different grid sizes**: How does S=7 vs S=4 affect performance?
2. **Add more anchor boxes**: Implement multiple aspect ratios per cell
3. **Real datasets**: Apply to PASCAL VOC or COCO datasets
4. **Evaluation metrics**: Implement mAP (mean Average Precision)
5. **Advanced architectures**: Study YOLOv5, YOLOv7, or YOLOv8
6. **Compare with R-CNN**: Implement Faster R-CNN and compare

YOLO revolutionized computer vision by making object detection fast enough for real-time applications. Understanding its core principles is essential for modern CV practitioners.